In [ ]:
import pandas as pd
%load_ext autoreload
%autoreload 2

from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import napari
import colorcet as cc

import dnt

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.size"] = 10
mpl.rcParams["axes.facecolor"] = (0.0, 0.0, 0.0, 0.0)

spots_directory = Path(r"C:\Tracking\BlastodermAnalysis\data\spots")
embryo_overview = pd.read_excel(spots_directory / "overview.xlsx", sheet_name="Sheet1")
save_path = Path(r"C:\Tracking\BlastodermAnalysis\figures\figure_3")


embryo_overview = embryo_overview[embryo_overview["good"]]
included = embryo_overview["Embryo"].astype(str).tolist()
condition_map = {
    str(embryo): condition for embryo, condition in zip(embryo_overview["Embryo"], embryo_overview["condition"])
}
print(condition_map)

dnt.set_plot_style()
spots_dfs, stems = dnt.load_spots_data(spots_directory, included)

print(stems)

df = spots_dfs[0]
cycles = [10, 11, 12, 13, 14]

print(df.columns)

greens = ["#143601","#1a4301","#245501","#538d22","#73a942","#aad576"][::-1]
blues = ["#012a4a","#01497c","#2a6f97","#468faf","#89c2d9"][::-1]
reds = ["#641220","#85182a","#a71e34","#bd1f36", "#da1e37"][::-1]
oranges = ["#fbba72","#ca5310","#bb4d00","#8f250c","#691e06"]
condition_pal_map = {
    "wt": blues,
    "bcd": reds,
    "trk": greens,
}
condition_main_colors = {
    c: cmap[2] for c, cmap in condition_pal_map.items()
}

condition_cycle_map = {}
for condition in condition_pal_map:
    for cycle in range(10, 15):
        condition_cycle_map[f"{condition}-{cycle}"] = condition_pal_map[condition][cycle-10]

# blender contour plot code

## Generate new meshes
If you run this, the blender meshes will be regenerated. This will require re-importing to blender and re-running the color assignment code below. You can set `remake_wt` and `remake_trk` to False to skip this step if you don't need to regenerate the meshes.

Note: i have no way of setting the random seed for the smoothed mesh generation, so the meshes will be different each time you run this. This will lead to different colors being assigned to the same points in blender, so if you want to keep the same colors, don't regenerate the meshes.

In [ ]:
nc = 11

remake_wt = False
wt_path = save_path.parent / "blender" / f"division_time_mesh_wt_smoothed{nc}.obj"
if remake_wt:
    division_times_df = dnt.division_times.get_division_times(spots_dfs[0])
    division_times_df = division_times_df.query("cycle == @nc").copy()

    positions = division_times_df[["z", "y", "x"]].values
    blender_mesh = dnt.smoothed_mesh_from_points(positions, 0.3)
    blender_mesh.write_obj(str(wt_path))

remake_trk = False
trk_path = save_path.parent / "blender" / f"division_time_mesh_trk2_smoothed{nc}.obj"
if remake_trk:
    division_times_df = dnt.division_times.get_division_times(spots_dfs[7])
    division_times_df = division_times_df.query("cycle == @nc").copy()

    positions = division_times_df[["z", "y", "x"]].values
    blender_mesh = dnt.smoothed_mesh_from_points(positions, 0.3)
    blender_mesh.write_obj(str(trk_path))

## Color Map

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

custom_csv = pd.read_csv(r"C:\Tracking\BlastodermAnalysis\figures\blender\ccc-tool_colormap_Custom CMS4.csv", delimiter=";", skiprows=0)

cmap = LinearSegmentedColormap.from_list("custom_cmap", [(0.0, "#0a9396"), (0.1, "#0a9396"), (0.3, "#7C974B"),  (0.5, "#ee9b00"), (0.7, "#CE5E09"), (0.9, "#ae2012"), (1.0, "#ae2012")], N=256)

cmap = LinearSegmentedColormap.from_list("custom_cmap", [(0.0, "#2F420C"),
                                                         (0.1, "#2F420C"),
                                                         (0.5, "#F6B946"),
                                                         (0.9, "#AE2012"),
                                                         (1.0, "#AE2012")], N=256)

cmap = LinearSegmentedColormap.from_list("custom_cmap", [(0.0, "#49D4E5"),
                                                         (0.02, "#49D4E5"),
                                                         (0.2, "#78D468"),
                                                         (0.5, "#EF9B47"),
                                                         (0.8, "#ED7E55"),
                                                         (0.98, "#E34236"),
                                                         (1.0, "#E34236")], N=256)

In [ ]:
max_below_mean = 0
max_above_mean = 7

# cmap = sns.color_palette("YlRd", as_cmap=True)
fig, ax = plt.subplots(1, 1, figsize=(0.6, 3))
norm = mpl.colors.Normalize(vmin=-max_below_mean, vmax=max_above_mean)

# Create the ScalarMappable and add the colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, cax=ax, location="right")
cbar.set_label("Division time")
plt.savefig(r"C:\Tracking\BlastodermAnalysis\figures\blender\wtcolorbar.pdf", dpi=300, bbox_inches="tight", format="pdf")
plt.show()


## Wild type

In [ ]:
from blender_tissue_cartography.mesh import ObjMesh
from scipy.spatial import KDTree

for nc in [11, 12, 13]:
    wt_path = save_path.parent / "blender" / f"division_time_mesh_wt_smoothed{nc}.obj"
    division_times_df = dnt.division_times.get_division_times(spots_dfs[0])
    division_times_df = division_times_df.query("cycle == @nc").copy()

    positions = division_times_df[["z", "y", "x"]].values
    new_positions = ObjMesh.read_obj(str(wt_path)).vertices

    t = division_times_df["corrected_division_time"].values

    # smooth using knn
    dists, indices = KDTree(positions).query(new_positions, k=25)

    t_new = []

    for i in range(len(new_positions)):
        neighbor_times = t[indices[i]]
        # skip exceptionally outlying neighbors
        is_valid = np.abs((neighbor_times - neighbor_times.mean())) / neighbor_times.std() < 2
        weights = np.exp(-dists[i] / 20)
        t_new.append(np.sum(neighbor_times[is_valid] * weights[is_valid]) / np.sum(weights[is_valid]))

    t = np.array(t_new)
    med = np.quantile(t, 0.01)


    t = np.clip(t, med - max_below_mean, med + max_above_mean)
    t_color = (t - (med - max_below_mean)) / (max_below_mean + max_above_mean)

    colors = cmap(t_color)
    time_groups = t // 0.25

    blender_csv = {
        "r": colors[:, 0],
        "g": colors[:, 1],
        "b": colors[:, 2],
        "time_group": time_groups.astype(int),
    }

    blender_df = pd.DataFrame(blender_csv)
    blender_df.to_csv(save_path.parent / "blender" / f"division_time_colors_wt_smoothed{nc}.csv", index=False)




In [ ]:
max_below_mean = 0
max_above_mean = 7

# cmap = sns.color_palette("YlRd", as_cmap=True)
fig, ax = plt.subplots(1, 1, figsize=(0.6, 3))
norm = mpl.colors.Normalize(vmin=-max_below_mean, vmax=max_above_mean)

# Create the ScalarMappable and add the colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, cax=ax, location="right")
cbar.set_label("Division time")
plt.savefig(r"C:\Tracking\BlastodermAnalysis\figures\blender\trkcolorbar.pdf", dpi=300, bbox_inches="tight", format="pdf")
plt.show()

In [ ]:
for nc in [11, 12, 13]:
    trk_path = save_path.parent / "blender" / f"division_time_mesh_trk2_smoothed{nc}.obj"

    division_times_df = dnt.division_times.get_division_times(spots_dfs[7])
    division_times_df = division_times_df.query("cycle == @nc").copy()

    positions = division_times_df[["z", "y", "x"]].values
    new_positions = ObjMesh.read_obj(str(trk_path)).vertices

    t = division_times_df["corrected_division_time"].values

    # smooth using knn
    dists, indices = KDTree(positions).query(new_positions, k=25)

    t_new = []

    for i in range(len(new_positions)):
        neighbor_times = t[indices[i]]
        # skip exceptionally outlying neighbors
        is_valid = np.abs((neighbor_times - neighbor_times.mean())) / neighbor_times.std() < 2
        weights = np.exp(-dists[i] / 20)
        t_new.append(np.sum(neighbor_times[is_valid] * weights[is_valid]) / np.sum(weights[is_valid]))

    t = np.array(t_new)
    med = np.quantile(t, 0.02)

    t_color_clipped = np.clip(t, med - max_below_mean, med + max_above_mean)
    t_color = (t_color_clipped - (med - max_below_mean)) / (max_below_mean + max_above_mean)

    colors = cmap(t_color)
    time_groups = t // 0.25

    blender_csv = {
        "r": colors[:, 0],
        "g": colors[:, 1],
        "b": colors[:, 2],
        "time_group": time_groups.astype(int),
    }


    blender_df = pd.DataFrame(blender_csv)
    blender_df.to_csv(save_path.parent / "blender" / f"division_time_colors_trk2_smoothed{nc}.csv", index=False)


# Division time scatterplots

In [ ]:
division_times_df = dnt.division_times.get_division_times(spots_dfs[0])
cycle_means = division_times_df.groupby("cycle")["corrected_division_time"].mean()
division_times_df["time_since_prev"] = division_times_df["corrected_division_time"] - (division_times_df["cycle"] - 1).map(cycle_means)

for cycle in [11, 12, 13]:
    fig, ax = plt.subplots(1, 1, figsize=(2.5, 1.5))

    cycle_df = division_times_df.query("cycle == @cycle and AP.between(0.0, 0.98)").copy()
    mean_time = cycle_df["time_since_prev"].quantile(0.001)
    cycle_df["relative_time"] = cycle_df["time_since_prev"] - mean_time

    # plt.axhline(0, color="k", linestyle="--", linewidth=2, zorder=-5)

    sns.scatterplot(cycle_df, x="AP", y="relative_time", alpha=1.0, edgecolor="k", color="#00265E", ax=ax, s=0.75*2**(14 - cycle), linewidth=0.1*1.5**(14 - cycle))


    ax.set_ylim(0, 8)
    ax.set_yticks([0, 4, 8])
    ax.set_xlim(0, 1)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_xlabel("AP position")
    # ax.spines["bottom"].set_visible(False)

    ax.set_ylabel("")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.savefig(save_path / f"division_time_scatter_wt_cycle{cycle}.pdf", dpi=300, bbox_inches="tight", format="pdf")
    plt.show()

In [ ]:
division_times_df = dnt.division_times.get_division_times(spots_dfs[7])
cycle_means = division_times_df.groupby("cycle")["corrected_division_time"].mean()
division_times_df["time_since_prev"] = division_times_df["corrected_division_time"] - (division_times_df["cycle"] - 1).map(cycle_means)

for cycle in [11, 12, 13]:
    fig, ax = plt.subplots(1, 1, figsize=(2.5, 1.5))

    cycle_df = division_times_df.query("cycle == @cycle and AP.between(0.0, 0.98)").copy()
    mean_time = cycle_df["time_since_prev"].quantile(0.002)
    cycle_df["relative_time"] = cycle_df["time_since_prev"] - mean_time

    # plt.axhline(0, color="k", linestyle="--", linewidth=2, zorder=-5)

    sns.scatterplot(cycle_df, x="AP", y="relative_time", alpha=1.0, edgecolor="k", color="#00265E", ax=ax, s=0.75*2**(14 - cycle), linewidth=0.1*1.5**(14 - cycle))


    ax.set_ylim(0, 8)
    ax.set_yticks([0, 4, 8])
    ax.set_xlim(0, 1)
    ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
    ax.set_xlabel("AP position")
    # ax.spines["bottom"].set_linestyle((0, (4, 4)))

    ax.set_ylabel("")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.savefig(save_path / f"division_time_scatter_trk_cycle{cycle}.pdf", dpi=300, bbox_inches="tight", format="pdf")
    plt.show()

In [ ]:
for k in [0, 1, 2, 3, 7, 8, 9, 10]:
    division_times_df = dnt.division_times.get_division_times(spots_dfs[k])
    cycle_means = division_times_df.groupby("cycle")["corrected_division_time"].mean()
    division_times_df["time_since_prev"] = division_times_df["corrected_division_time"] - (division_times_df["cycle"] - 1).map(cycle_means)

    for cycle in [11, 12, 13]:
        fig, ax = plt.subplots(1, 1, figsize=(2.5, 1.5))

        cycle_df = division_times_df.query("cycle == @cycle and AP.between(0.0, 0.98)").copy()
        mean_time = cycle_df["time_since_prev"].quantile(0.001)
        cycle_df["relative_time"] = cycle_df["time_since_prev"] - mean_time

        # plt.axhline(0, color="k", linestyle="--", linewidth=2, zorder=-5)

        sns.scatterplot(cycle_df, x="AP", y="relative_time", alpha=1.0, edgecolor="k", color="#00265E", ax=ax, s=0.75*2**(14 - cycle), linewidth=0.1*1.5**(14 - cycle))


        ax.set_ylim(0, 8)
        ax.set_yticks([0, 4, 8])
        ax.set_xlim(0, 1)
        ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
        ax.set_xlabel("AP position")
        # ax.spines["bottom"].set_visible(False)

        ax.set_ylabel("")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        plt.savefig(save_path / f"division_time_scatter_{stems[k]}_cycle{cycle}.png", dpi=300, bbox_inches="tight")
        plt.show()